# Bayesian Reorder-Rate Estimator (Instacart) — from the posterior CSV

Estimates, for every product, the probability that an order line is a **reorder**, with a 95% credible interval.

This version works **only from `product_reorder_posterior.csv`**. The raw Instacart files are not needed.

**Model (Beta-Binomial, empirical Bayes)**

- Data per product: `n` = times ordered, `k` = times reordered.
- Prior: `Beta(K * p_aisle, K * (1 - p_aisle))`, where `p_aisle` is the volume-weighted reorder rate of the product's aisle and `K` is the prior strength (pseudo-orders).
- Posterior: `Beta(alpha0 + k, beta0 + n - k)`.
- Outputs: posterior mean, 95% equal-tailed credible interval, interval width.

Low-volume products are pulled toward their aisle average (shrinkage). High-volume products stay close to their raw rate.

**File expected in `DATA_FILE`:** `product_reorder_posterior.csv`, with columns `product_id, product_name, aisle, department, times_ordered, times_reordered, aisle_reorder_rate, prior_alpha, prior_beta, post_alpha, post_beta, raw_rate, posterior_mean, ci_lower, ci_upper, ci_width`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (8, 5)

# ----------------------------- Configuration -----------------------------
DATA_FILE = Path(os.environ.get("DATA_FILE", "product_reorder_posterior.csv"))
OUT_DIR = Path(os.environ.get("OUT_DIR", "output"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

CRED_MASS = 0.95         # credible interval mass
LOW_DATA_THRESHOLD = 10  # products with fewer orders are flagged as low-data

## 1. Load data

In [ ]:
df = pd.read_csv(DATA_FILE)
df["times_reordered"] = df["times_reordered"].astype("int64")

required = ["product_id", "product_name", "aisle", "department", "times_ordered", "times_reordered",
            "aisle_reorder_rate", "prior_alpha", "prior_beta", "post_alpha", "post_beta",
            "raw_rate", "posterior_mean", "ci_lower", "ci_upper", "ci_width"]
missing = [col for col in required if col not in df.columns]
assert not missing, f"Missing columns: {missing}"

# Prior strength K is stored implicitly as prior_alpha + prior_beta
PRIOR_STRENGTH = float(round((df["prior_alpha"] + df["prior_beta"]).median()))
assert np.allclose(df["prior_alpha"] + df["prior_beta"], PRIOR_STRENGTH, atol=1e-3), "K is not constant"

print(f"Products loaded      : {len(df):,}")
print(f"Prior strength K     : {PRIOR_STRENGTH:.0f}")
print(f"Aisles / departments : {df['aisle'].nunique()} / {df['department'].nunique()}")

## 2. Product table with raw reorder rate

In [ ]:
# Recompute the raw rate from the counts rather than trusting the stored column
df["raw_rate"] = df["times_reordered"] / df["times_ordered"]

print(f"Total product orders : {df['times_ordered'].sum():,}")
print(f"Overall reorder rate : {df['times_reordered'].sum() / df['times_ordered'].sum():.4f}")
print(f"Missing names        : {df['product_name'].isna().sum()}")
df.head()

## 3. Empirical-Bayes prior (per aisle)

The prior mean is the aisle's **volume-weighted** reorder rate (total reorders divided by total orders). Averaging per-product rates would let 1-order products with a rate of 0 or 1 distort it.

The CSV already carries `aisle_reorder_rate`. Here it is recomputed from the counts and checked against the stored value.

In [ ]:
aisle_stats = df.groupby("aisle")[["times_reordered", "times_ordered"]].sum()
aisle_stats["aisle_rate"] = aisle_stats["times_reordered"] / aisle_stats["times_ordered"]

stored = df.groupby("aisle")["aisle_reorder_rate"].first()
print(f"Max |recomputed - stored| aisle rate: {(aisle_stats['aisle_rate'] - stored).abs().max():.2e}")

df = df.drop(columns=["aisle_reorder_rate"]).merge(
    aisle_stats[["aisle_rate"]], left_on="aisle", right_index=True, how="left")

df["prior_alpha"] = PRIOR_STRENGTH * df["aisle_rate"]
df["prior_beta"] = PRIOR_STRENGTH * (1 - df["aisle_rate"])

df[["aisle", "aisle_rate", "prior_alpha", "prior_beta"]].drop_duplicates("aisle").head()

## 4. Posterior and credible interval

Because the prior is conjugate, the posterior is available in closed form, with no MCMC needed. The interval is **equal-tailed** (2.5th and 97.5th percentiles), not an HDI.

The posterior is recomputed from the counts and compared with the values stored in the CSV.

In [ ]:
tail = (1 - CRED_MASS) / 2
stored_post = df[["posterior_mean", "ci_lower", "ci_upper"]].copy()

df["post_alpha"] = df["prior_alpha"] + df["times_reordered"]
df["post_beta"] = df["prior_beta"] + (df["times_ordered"] - df["times_reordered"])

df["posterior_mean"] = df["post_alpha"] / (df["post_alpha"] + df["post_beta"])
df["ci_lower"] = stats.beta.ppf(tail, df["post_alpha"], df["post_beta"])
df["ci_upper"] = stats.beta.ppf(1 - tail, df["post_alpha"], df["post_beta"])
df["ci_width"] = df["ci_upper"] - df["ci_lower"]
df["low_data"] = df["times_ordered"] < LOW_DATA_THRESHOLD

# Agreement with the CSV (CSV values are rounded to 4 decimals)
diff = (df[["posterior_mean", "ci_lower", "ci_upper"]] - stored_post).abs().max()
print("Max abs difference vs stored CSV values:")
print(diff)
assert (diff < 1e-3).all(), "Recomputed posterior disagrees with the CSV"

df[["product_name", "aisle", "times_ordered", "raw_rate",
    "posterior_mean", "ci_lower", "ci_upper", "ci_width"]].sample(8, random_state=1)

## 5. Sanity checks

In [ ]:
# 1) The posterior mean must lie between the raw rate and the prior mean (shrinkage property)
lo = np.minimum(df["raw_rate"], df["aisle_rate"]) - 1e-9
hi = np.maximum(df["raw_rate"], df["aisle_rate"]) + 1e-9
assert df["posterior_mean"].between(lo, hi).all(), "Posterior mean outside [raw, prior]"

# 2) Intervals are valid and contain the posterior mean
assert (df["ci_lower"] <= df["posterior_mean"]).all() and (df["posterior_mean"] <= df["ci_upper"]).all()
assert df[["posterior_mean", "ci_lower", "ci_upper"]].notna().all().all()

# 3) More data means a narrower interval (compare low- vs high-volume products)
q = df.groupby(pd.qcut(df["times_ordered"], 5, duplicates="drop"), observed=True)["ci_width"].mean()
print("Mean interval width by order-volume quintile:")
print(q)
assert q.is_monotonic_decreasing

print(f"\nLow-data products (<{LOW_DATA_THRESHOLD} orders): {df['low_data'].sum():,} "
      f"({df['low_data'].mean():.1%})")
print("All checks passed.")

## 6. Visuals

- **Left:** uncertainty falls as order volume grows (log x-axis).
- **Right:** raw vs posterior rate. Points off the diagonal are the shrunk low-volume products.

In [ ]:
sample = df.sample(min(8000, len(df)), random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(sample["times_ordered"], sample["ci_width"], s=6, alpha=0.4)
axes[0].set_xscale("log")
axes[0].set_xlabel("Times ordered (log scale)")
axes[0].set_ylabel("95% interval width")
axes[0].set_title("Uncertainty shrinks as order volume grows")

sc = axes[1].scatter(sample["raw_rate"], sample["posterior_mean"], s=6, alpha=0.4,
                     c=np.log10(sample["times_ordered"]), cmap="viridis")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("Raw reorder rate")
axes[1].set_ylabel("Posterior mean")
axes[1].set_title("Raw vs posterior reorder rate")
fig.colorbar(sc, ax=axes[1], label="log10(times ordered)")

plt.tight_layout()
plt.savefig(OUT_DIR / "shrinkage_overview.png", dpi=150)
plt.show()

In [ ]:
# Interval for high-volume products, sorted by posterior mean
top = (df[df["times_ordered"] >= 1000]
       .nlargest(15, "posterior_mean")
       .sort_values("posterior_mean"))

fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(top["posterior_mean"], range(len(top)),
            xerr=[top["posterior_mean"] - top["ci_lower"], top["ci_upper"] - top["posterior_mean"]],
            fmt="o", capsize=3)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top["product_name"].str.slice(0, 35))
ax.set_xlabel("Posterior reorder rate (95% credible interval)")
ax.set_title("Top 15 products by posterior reorder rate (at least 1,000 orders)")
plt.tight_layout()
plt.savefig(OUT_DIR / "top_products_intervals.png", dpi=150)
plt.show()

## 7. Prior-strength sensitivity

How much does the choice of `K` change the estimate for a product with little data? The table shows a product with 1 order and 0 reorders in an aisle with a 60% reorder rate.

In [ ]:
def posterior_summary(n, k, p_prior, K, mass=CRED_MASS):
    a, b = K * p_prior + k, K * (1 - p_prior) + (n - k)
    t = (1 - mass) / 2
    lo, hi = stats.beta.ppf([t, 1 - t], a, b)
    return {"K": K, "posterior_mean": a / (a + b), "ci_lower": lo, "ci_upper": hi, "ci_width": hi - lo}

pd.DataFrame([posterior_summary(n=1, k=0, p_prior=0.60, K=K) for K in [2, 5, 10, 20, 50, 100]])

## 8. Export for Power BI and the API

Writes three tables: product-level, aisle-level, and department-level. Aisle and department rates are volume-weighted.

In [ ]:
export_cols = ["product_id", "product_name", "aisle", "department",
               "times_ordered", "times_reordered", "raw_rate",
               "prior_alpha", "prior_beta", "post_alpha", "post_beta",
               "posterior_mean", "ci_lower", "ci_upper", "ci_width", "low_data"]
df[export_cols].to_csv(OUT_DIR / "product_reorder_posteriors.csv", index=False)

def group_summary(frame, by):
    g = frame.groupby(by)[["times_ordered", "times_reordered"]].sum().reset_index()
    g["raw_rate"] = g["times_reordered"] / g["times_ordered"]
    g["post_alpha"] = PRIOR_STRENGTH * 0.5 + g["times_reordered"]
    g["post_beta"] = PRIOR_STRENGTH * 0.5 + (g["times_ordered"] - g["times_reordered"])
    g["posterior_mean"] = g["post_alpha"] / (g["post_alpha"] + g["post_beta"])
    g["ci_lower"] = stats.beta.ppf(tail, g["post_alpha"], g["post_beta"])
    g["ci_upper"] = stats.beta.ppf(1 - tail, g["post_alpha"], g["post_beta"])
    return g.drop(columns=["post_alpha", "post_beta"])

group_summary(df, "aisle").to_csv(OUT_DIR / "aisle_reorder_posteriors.csv", index=False)
group_summary(df, "department").to_csv(OUT_DIR / "department_reorder_posteriors.csv", index=False)

print("Saved to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)

## Notes and limitations

- This notebook starts from the already-aggregated CSV, so it cannot change which order files were used (prior and/or train). Counts are taken as given.
- The prior strength `K` is read from the CSV (`prior_alpha + prior_beta`), not learned. Section 7 shows the sensitivity. `K` could instead be fitted by maximizing the Beta-Binomial marginal likelihood across products.
- Products are modeled independently, so seasonality and customer-level behavior are not captured.
- The aisle and department tables use a flat Beta(10, 10) prior. With thousands of orders per group, the prior has almost no effect.
- Intervals are equal-tailed credible intervals. For a Beta posterior far from 0.5, an HDI would be slightly narrower.